In [1]:
from pathlib import Path
import pandas as pd
import subprocess

def find_csv_files(base_dir):
    base_path = Path(base_dir)
    csv_files = []
    for path in base_path.rglob("TOIs_in*.csv"):
        # Exclude any path that is inside a hidden directory
        idx = any(part.startswith('.ipynb') or part.endswith('_merged.csv') for part in path.parts)
        if not idx:
            csv_files.append(path)

    return csv_files

# Example usage
csv_files = find_csv_files("../data/")

# Print results
dfs = {}
for f in csv_files:
    #print(f)
    subprocess.run(["ls", "-lt", f])
    n = str(f).split('/')[2]
    d = pd.read_csv(f)
    d['catalog'] = n
    dfs[n] = d

-rw-r--r-- 1 jerome muscat 43934 Nov 21 12:13 ../data/CantatGaudin2020/TOIs_in_CantatGaudin2020.csv
-rw-r--r-- 1 jerome muscat 21399 Jan 16 10:06 ../data/Risbud2025/TOIs_in_Risbud2025.csv
-rw-r--r-- 1 jerome muscat 90011 May 15  2025 ../data/Hunt2023/TOIs_in_Hunt2023.csv
-rw-r--r-- 1 jerome muscat 91976 Nov 21 12:12 ../data/Hunt2024/TOIs_in_Hunt2024.csv
-rw-r--r-- 1 jerome muscat 36193 Sep 26  2024 ../data/Kounkel2020/TOIs_in_Kounkel2020.csv
-rw-rw-r-- 1 jerome muscat 77922 Jan 16 10:04 ../data/Perren2023/TOIs_in_Perren2023.csv
-rw-r--r-- 1 jerome muscat 11585 Nov 21 12:19 ../data/Ratzenboeck2023a/TOIs_in_Ratzenboeck2023a.csv


In [2]:
df_concat = pd.concat(dfs, ignore_index=True)
catalog_duplicates = df_concat.groupby('TOI')['catalog'].unique().apply(lambda x: ", ".join(map(str, x)))
df_concat = df_concat.drop(columns=['catalog'])
df_unique = df_concat.drop_duplicates(subset='TOI', keep='first')
# add catalog column to df_unique from catalog_duplicates
df_final = df_unique.merge(catalog_duplicates, on="TOI", how="left")
df_final = df_final.sort_values(by='TOI', ascending=False)
df_final.head()

,TOI,TIC ID,Name,GaiaDR3,Prob,TESS Disposition,TFOPWG Disposition,Comments,TESS Mag,TESS Mag err,...,PlxpscolCorr,pmRApscolCorr,pmDEpscolCorr,BPmag,RPmag,BP-RP_y,RV,OName,logAge,catalog
26,7497.01,130633912,Ruprecht_147,4087782677845919744,1.000000,KP,KP,brown dwarf EPIC 219388192 b,11.8543,0.006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"CantatGaudin2020, Risbud2025, Hunt2024, Perren..."
40,7111.01,200465781,Alessi_37,1983367454361525888,0.626075,PC,PC,found in faint-star QLP search,12.9436,0.012,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Hunt2023, Hunt2024"
61,7038.01,210904767,HSC_2907,6025147218543620480,0.757498,PC,PC,variable host,8.9498,0.006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Hunt2023, Hunt2024, Ratzenboeck2023a"
44,6963.01,56658270,CWNU_1129,164800235906366976,1.000000,PC,CP,T Tauri host,12.8131,0.007,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Hunt2023, Hunt2024"
31,6940.01,421006834,Melotte_25,3162043644458156416,NaN,PC,PC,found in faint-star QLP search,12.5437,0.007,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Risbud2025


In [3]:
idx = df_final['TFOPWG Disposition'].isin(['KP','CP','EB','FA','FP'])
df_final = df_final[~idx]
df_final

,TOI,TIC ID,Name,GaiaDR3,Prob,TESS Disposition,TFOPWG Disposition,Comments,TESS Mag,TESS Mag err,...,PlxpscolCorr,pmRApscolCorr,pmDEpscolCorr,BPmag,RPmag,BP-RP_y,RV,OName,logAge,catalog
40,7111.01,200465781,Alessi_37,1983367454361525888,0.626075,PC,PC,found in faint-star QLP search,12.94360,0.012,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Hunt2023, Hunt2024"
61,7038.01,210904767,HSC_2907,6025147218543620480,0.757498,PC,PC,variable host,8.94980,0.006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Hunt2023, Hunt2024, Ratzenboeck2023a"
31,6940.01,421006834,Melotte_25,3162043644458156416,NaN,PC,PC,found in faint-star QLP search,12.54370,0.007,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Risbud2025
9,6715.01,460950389,IC_2602,5239758155778687360,1.000000,PC,PC,host is a BY Dra variable star; low MES,11.64900,0.006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"CantatGaudin2020, Hunt2023, Hunt2024, Kounkel2..."
60,6705.01,379717270,HSC_2675,5895008644342480640,1.000000,PC,PC,variable host; multiple stars in pixel,10.27340,0.006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Hunt2023, Hunt2024, Perren2023"
82,6697.02,441546821,NaN,6055854551117476480,NaN,PC,PC,possible additional planet in HD 114082 star s...,7.78970,0.006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ratzenboeck2023a
3,6650.04,64837857,ASCC_123,2003378188041736320,1.000000,PC,PC,single transit at TBJD ~2878; potential multi,10.02520,0.006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"CantatGaudin2020, Hunt2023, Hunt2024, Perren2023"
2,6650.03,64837857,ASCC_123,2003378188041736320,1.000000,PC,PC,potential multi,10.02520,0.006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"CantatGaudin2020, Hunt2023, Hunt2024, Perren2023"
1,6650.02,64837857,ASCC_123,2003378188041736320,1.000000,PC,PC,potential multi near 3:2 resonance,10.02520,0.006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"CantatGaudin2020, Hunt2023, Hunt2024, Perren2023"
0,6650.01,64837857,ASCC_123,2003378188041736320,1.000000,PC,PC,potential multi; possible additional stellar o...,10.02520,0.006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"CantatGaudin2020, Hunt2023, Hunt2024, Kounkel2..."


In [4]:
df_final.to_csv('../data/TOIs_in_cluster_merged.csv', index=False)
cols = ['TOI', 'TIC ID', 'catalog', 'Prob', 'TFOPWG Disposition', 'Comments', 'TESS Mag', 'logAge', 
        'Period (days)'
       ]
df_final[cols]

,TOI,TIC ID,catalog,Prob,TFOPWG Disposition,Comments,TESS Mag,logAge,Period (days)
40,7111.01,200465781,"Hunt2023, Hunt2024",0.626075,PC,found in faint-star QLP search,12.94360,NaN,7.063428
61,7038.01,210904767,"Hunt2023, Hunt2024, Ratzenboeck2023a",0.757498,PC,variable host,8.94980,NaN,8.600049
31,6940.01,421006834,Risbud2025,NaN,PC,found in faint-star QLP search,12.54370,NaN,0.305261
9,6715.01,460950389,"CantatGaudin2020, Hunt2023, Hunt2024, Kounkel2...",1.000000,PC,host is a BY Dra variable star; low MES,11.64900,NaN,2.861878
60,6705.01,379717270,"Hunt2023, Hunt2024, Perren2023",1.000000,PC,variable host; multiple stars in pixel,10.27340,NaN,8.445704
82,6697.02,441546821,Ratzenboeck2023a,NaN,PC,possible additional planet in HD 114082 star s...,7.78970,NaN,0.000000
3,6650.04,64837857,"CantatGaudin2020, Hunt2023, Hunt2024, Perren2023",1.000000,PC,single transit at TBJD ~2878; potential multi,10.02520,NaN,0.000000
2,6650.03,64837857,"CantatGaudin2020, Hunt2023, Hunt2024, Perren2023",1.000000,PC,potential multi,10.02520,NaN,8.996513
1,6650.02,64837857,"CantatGaudin2020, Hunt2023, Hunt2024, Perren2023",1.000000,PC,potential multi near 3:2 resonance,10.02520,NaN,30.505563
0,6650.01,64837857,"CantatGaudin2020, Hunt2023, Hunt2024, Kounkel2...",1.000000,PC,potential multi; possible additional stellar o...,10.02520,NaN,20.329213


In [5]:
df_final.columns.tolist()

['TOI',
 'TIC ID',
 'Name',
 'GaiaDR3',
 'Prob',
 'TESS Disposition',
 'TFOPWG Disposition',
 'Comments',
 'TESS Mag',
 'TESS Mag err',
 'Source_1',
 'Detection',
 'Epoch (BJD)',
 'Epoch (BJD) err',
 'Period (days)',
 'Period (days) err',
 'Duration (hours)',
 'Duration (hours) err',
 'Depth (mmag)',
 'Depth (mmag) err',
 'Depth (ppm)',
 'Depth (ppm) err',
 'Planet Radius (R_Earth)',
 'Planet Radius (R_Earth) err',
 'Planet Insolation (Earth Flux)',
 'Planet Equil Temp (K)',
 'Planet SNR',
 'Stellar Distance (pc)',
 'Stellar Distance (pc) err',
 'Stellar Eff Temp (K)',
 'Stellar Eff Temp (K) err',
 'Stellar log(g) (cm/s^2)',
 'Stellar log(g) (cm/s^2) err',
 'Stellar Radius (R_Sun)',
 'Stellar Radius (R_Sun) err',
 'Stellar Metallicity',
 'Stellar Metallicity err',
 'Stellar Mass (M_Sun)',
 'Stellar Mass (M_Sun) err',
 'Sectors',
 'Date TOI Alerted (UTC)',
 'Date TOI Updated (UTC)',
 'Date Modified',
 'ra_deg',
 'dec_deg',
 'Pipeline Signal ID',
 'RA',
 'Dec',
 'PM RA (mas/yr)',
 'PM RA